# BLIP-2 zero-shot captioning on ROCOv2 — reusing the VQA-RAD (LoRA-ViT) model

**task:** *use the model we fine-tuned on VQA-RAD for the first ROCOv2
inference run, then further fine-tune it on a subset.* This notebook is **step 1:
zero-shot captioning** (no ROCO training yet).

**Key idea — captioning is VQA with the question removed.** Our `Blip2QFormerVQA`
feeds `[query_tokens, question]` into the Q-Former. If we pass **no question**, the
Q-Former runs the **query-only** path — exactly the standard BLIP-2 captioning setup:
the 32 learned queries attend to the image, the LLM is primed with a short caption
prompt (*"a photo of"*), and completes it. Same weights, same network; we just drop
the question. (`generate_caption` below is the only new method vs the VQA notebook.)

We load the LoRA-ViT checkpoint (`vqa_lora_final.pt`), caption the ROCOv2 test
images, and score against the reference captions with **BLEU-1..4, METEOR, ROUGE-L,
CIDEr and BERTScore**.

> Cells 1–4 (setup / load BLIP-2 / model class / instantiate) are copied verbatim
> from `blip-2_fine_tuned_VQA-RAD_LoRA-ViT.ipynb` so the model is byte-for-byte the
> same — only the query-only captioning path and the ROCO data/metrics are new.

In [5]:
import torch
import torch.nn as nn
from PIL import Image

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
# bfloat16: same memory as fp16 but fp32's dynamic range (matches the training run).
dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32
print(f"device = {device} | dtype = {dtype}")

device = cuda | dtype = torch.bfloat16


In [6]:
# ── Load BLIP-2 and the Q-Former question-embedding table (same as the VQA notebook) ──
from typing import Any
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BertTokenizer

CKPT = "/home/matei/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(CKPT)
blip2 = Blip2ForConditionalGeneration.from_pretrained(CKPT, torch_dtype=dtype, low_cpu_mem_usage=True)

# The checkpoint we load below was trained WITH the grafted Q-Former text weights, so
# we must rebuild the same architecture (text FFN + word/pos embeddings) before loading,
# even though captioning itself never uses the question path.
qformer_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
_QF_TEXT = torch.load("/home/matei/blip2_qformer_text_weights.pt", map_location="cpu")
qformer_word_emb = nn.Embedding.from_pretrained(_QF_TEXT["word_embeddings.weight"], freeze=False)
qformer_pos_emb  = nn.Embedding.from_pretrained(_QF_TEXT["position_embeddings.weight"], freeze=False)
qformer_text_ffn = _QF_TEXT["text_ffn"]

cfg: Any = blip2.config
d_llm     = cfg.text_config.hidden_size
d_qformer = cfg.qformer_config.hidden_size
n_query   = cfg.num_query_tokens
print(f"d_llm={d_llm} | d_qformer={d_qformer} | num_query_tokens={n_query}")

Loading weights: 100%|██████████| 1247/1247 [00:01<00:00, 959.90it/s] 


d_llm=2560 | d_qformer=768 | num_query_tokens=32


/tmp/ipykernel_72193/236649605.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _QF_TEXT = torch.load("/home/matei/blip2_qformer_text_weights.pt", map_location="cpu")


## The model — VQA path + a new query-only captioning path

`_qformer_features` now takes `q_ids=None`. With a question it runs the VQA path
(queries **+** question tokens through the Q-Former, Section 4.3). With `q_ids=None`
it runs the **captioning path**: queries only — the standard BLIP-2 setup. Everything
else (cross-attention to the image, projection to the LLM width) is unchanged.

`generate_caption` is the new inference method: query-only features + an optional
short caption prompt to the LLM, then decode. `forward`/`generate` (the VQA methods)
are kept unchanged so the checkpoint loads with an identical architecture.

In [7]:
import copy

class Blip2QFormerVQA(nn.Module):
    # VQA-RAD model. Captioning = call the query-only path (q_ids=None).
    def __init__(self, blip2, word_emb, pos_emb, text_ffn=None):
        super().__init__()
        self.vision_model        = blip2.vision_model
        self.query_tokens        = blip2.query_tokens
        self.qformer             = blip2.qformer
        for _i, _layer in enumerate(self.qformer.encoder.layer):
            if not hasattr(_layer, 'intermediate'):
                _layer.intermediate = copy.deepcopy(_layer.intermediate_query)
                _layer.output       = copy.deepcopy(_layer.output_query)
                if text_ffn is not None:
                    _layer.intermediate.load_state_dict(text_ffn[_i]['intermediate'])
                    _layer.output.load_state_dict(text_ffn[_i]['output'])
        self.language_projection = blip2.language_projection
        self.language_model      = blip2.language_model
        self.qformer_word_emb    = word_emb
        self.qformer_pos_emb     = pos_emb
        self.n_query             = blip2.config.num_query_tokens

    def _qformer_features(self, pixel_values, q_ids=None, q_att=None):
        image_embeds = self.vision_model(pixel_values).last_hidden_state
        image_atts = torch.ones(image_embeds.shape[:-1], dtype=torch.long, device=image_embeds.device)
        B = pixel_values.shape[0]
        query_tokens = self.query_tokens.expand(B, -1, -1)
        query_atts   = torch.ones(query_tokens.shape[:-1], dtype=torch.long, device=query_tokens.device)

        if q_ids is not None:
            # VQA path: concatenate the question tokens after the 32 queries
            seq_len  = q_ids.shape[1]
            pos_ids  = torch.arange(seq_len, device=q_ids.device).unsqueeze(0)
            text_emb = (self.qformer_word_emb(q_ids) + self.qformer_pos_emb(pos_ids)).to(query_tokens.dtype)
            query_embeds   = torch.cat([query_tokens, text_emb], dim=1)
            attention_mask = torch.cat([query_atts, q_att], dim=1)
        else:
            # Captioning path: queries only (standard BLIP-2 captioning)
            query_embeds   = query_tokens
            attention_mask = query_atts

        out = self.qformer(
            query_embeds=query_embeds,
            query_length=self.n_query,
            attention_mask=attention_mask,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_atts,
        )
        query_output = out.last_hidden_state[:, :self.n_query, :].to(self.language_projection.weight.dtype)
        return self.language_projection(query_output)

    def forward(self, pixel_values, q_ids, q_att, llm_ids, llm_att, labels):
        soft = self._qformer_features(pixel_values, q_ids, q_att)
        soft_att = torch.ones(soft.shape[:-1], dtype=torch.long, device=soft.device)
        text_embeds   = self.language_model.get_input_embeddings()(llm_ids)
        inputs_embeds = torch.cat([soft, text_embeds], dim=1)
        attention_mask = torch.cat([soft_att, llm_att], dim=1)
        B = pixel_values.shape[0]
        soft_labels = torch.full((B, self.n_query), -100, dtype=labels.dtype, device=labels.device)
        full_labels = torch.cat([soft_labels, labels], dim=1)
        return self.language_model(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=full_labels)

    @torch.no_grad()
    def generate(self, pixel_values, q_ids, q_att, llm_ids, llm_att, **gen_kwargs):
        soft = self._qformer_features(pixel_values, q_ids, q_att)
        soft_att = torch.ones(soft.shape[:-1], dtype=torch.long, device=soft.device)
        text_embeds   = self.language_model.get_input_embeddings()(llm_ids)
        inputs_embeds = torch.cat([soft, text_embeds], dim=1)
        attention_mask = torch.cat([soft_att, llm_att], dim=1)
        return self.language_model.generate(inputs_embeds=inputs_embeds, attention_mask=attention_mask, **gen_kwargs)

    @torch.no_grad()
    def generate_caption(self, pixel_values, prompt_ids=None, prompt_att=None, **gen_kwargs):
        # Query-only captioning. Optional short prompt (e.g. "a photo of") primes the LLM.
        soft = self._qformer_features(pixel_values)                     # query-only
        soft_att = torch.ones(soft.shape[:-1], dtype=torch.long, device=soft.device)
        if prompt_ids is not None:
            text_embeds    = self.language_model.get_input_embeddings()(prompt_ids)
            inputs_embeds  = torch.cat([soft, text_embeds], dim=1)
            attention_mask = torch.cat([soft_att, prompt_att], dim=1)
        else:
            inputs_embeds, attention_mask = soft, soft_att
        return self.language_model.generate(inputs_embeds=inputs_embeds, attention_mask=attention_mask, **gen_kwargs)

In [8]:
# ── Instantiate, re-inject the ViT LoRA adapters, load the VQA-RAD checkpoint ──
from peft import LoraConfig, inject_adapter_in_model

model = Blip2QFormerVQA(blip2, qformer_word_emb, qformer_pos_emb, qformer_text_ffn)

# The checkpoint stored ViT LoRA adapters, so recreate the SAME adapter structure
# (identical config to training) before loading the weights into it.
for p in model.vision_model.parameters():
    p.requires_grad = False
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=["qkv", "projection", "fc1", "fc2"], bias="none")
inject_adapter_in_model(lora_cfg, model.vision_model)

model = model.to(device)
model.qformer_word_emb = model.qformer_word_emb.to(device, model.query_tokens.dtype)
model.qformer_pos_emb  = model.qformer_pos_emb.to(device, model.query_tokens.dtype)

LOAD_PATH = "/home/matei/vqa_checkpoints_lora/vqa_lora_final.pt"
ckpt = torch.load(LOAD_PATH, map_location=device)
model.qformer.load_state_dict(ckpt["qformer"])
model.language_projection.load_state_dict(ckpt["language_projection"])
with torch.no_grad():
    model.query_tokens.copy_(ckpt["query_tokens"].to(device, model.query_tokens.dtype))
model.qformer_word_emb.load_state_dict(ckpt["qformer_word_emb"])
model.qformer_pos_emb.load_state_dict(ckpt["qformer_pos_emb"])
model.vision_model.load_state_dict(ckpt["vit_lora"], strict=False)   # LoRA adapters only
model.eval()
print(f"loaded VQA-RAD (LoRA-ViT) weights from {LOAD_PATH}")

/tmp/ipykernel_72193/3439883884.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(LOAD_PATH, map_location=device)


loaded VQA-RAD (LoRA-ViT) weights from /home/matei/vqa_checkpoints_lora/vqa_lora_final.pt


In [9]:
# ── Load the ROCOv2 test split (image -> reference caption) ──
import os, pandas as pd

ROCO_DIR = "/home/matei/rocov2"
IMG_DIR  = os.path.join(ROCO_DIR, "test")
caps = pd.read_csv(os.path.join(ROCO_DIR, "test_captions.csv"))   # columns: ID, Caption
caps = caps.dropna(subset=["Caption"]).reset_index(drop=True)

def roco_image_path(_id):
    return os.path.join(IMG_DIR, f"{_id}.jpg")

records = [{"id": r.ID, "path": roco_image_path(r.ID), "caption": str(r.Caption)}
           for r in caps.itertuples() if os.path.isfile(roco_image_path(r.ID))]
print(f"ROCOv2 test: {len(records)} image-caption pairs")
print("example:", records[0]["id"], "->", records[0]["caption"][:80])

ROCOv2 test: 9927 image-caption pairs
example: ROCOv2_2023_test_000001 -> CT chest axial view showing a huge ascending aortic aneurysm (*).


In [10]:
# ── Captioning function (query-only, batched: same prompt -> no padding needed) ──
from tqdm.auto import tqdm

CAPTION_PROMPT = "a photo of"      # BLIP-2's default captioning prompt
CAPTION_BATCH  = 8
GEN_KWARGS = dict(max_new_tokens=40, min_new_tokens=8,
                  num_beams=5, no_repeat_ngram_size=3, length_penalty=1.0,
                  eos_token_id=processor.tokenizer.eos_token_id,
                  pad_token_id=processor.tokenizer.eos_token_id)

def _load_pixels(paths):
    imgs = [Image.open(p).convert("RGB") for p in paths]
    return processor(images=imgs, return_tensors="pt").pixel_values.to(device, dtype)  # pyright: ignore[reportCallIssue]

def caption_records(recs, prompt=CAPTION_PROMPT, batch_size=CAPTION_BATCH, **gen_kwargs):
    gk = {**GEN_KWARGS, **gen_kwargs}
    base = processor.tokenizer(prompt, return_tensors="pt").to(device) if prompt else None
    preds = []
    for i in tqdm(range(0, len(recs), batch_size), desc="captioning"):
        chunk = recs[i:i+batch_size]
        pix = _load_pixels([r["path"] for r in chunk])
        B = pix.shape[0]
        if base is not None:
            pid, pat = base.input_ids.expand(B, -1), base.attention_mask.expand(B, -1)
        else:
            pid = pat = None
        gen = model.generate_caption(pix, pid, pat, **gk)
        preds.extend(s.strip() for s in processor.tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds

In [11]:
# ── SANITY CHECK: what does the model actually caption? Compare 3 strategies ──
# This is the important first look -- our model was tuned to answer VQA in 1-2 words,
# so we check whether it produces caption-like text and which prompting works best.
sample = records[:8]

p1 = caption_records(sample, prompt="a photo of", batch_size=8)   # query-only + prompt (standard captioning)
p2 = caption_records(sample, prompt="",           batch_size=8)   # query-only + no prompt

# 3rd option: feed a "describe" question through the full VQA path (in-distribution
# for our fine-tuned model, since it was always trained on "Question: ... Answer:").
def caption_via_question(recs, question="describe the image.", batch_size=8, **gk):
    g = {**GEN_KWARGS, **gk}
    preds = []
    for i in range(0, len(recs), batch_size):
        chunk = recs[i:i+batch_size]
        pix = _load_pixels([r["path"] for r in chunk])
        qf  = qformer_tokenizer([question]*len(chunk), padding="max_length", truncation=True,
                                max_length=32, return_tensors="pt").to(device)
        pr  = processor.tokenizer([f"Question: {question} Answer:"]*len(chunk),
                                  return_tensors="pt", padding=True).to(device)
        gen = model.generate(pix, qf.input_ids, qf.attention_mask, pr.input_ids, pr.attention_mask, **g)
        preds.extend(s.strip() for s in processor.tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds
p3 = caption_via_question(sample)

for i, r in enumerate(sample):
    print(f"[{r['id']}]")
    print(f"  REF                     : {r['caption']}")
    print(f"  query-only 'a photo of' : {p1[i]}")
    print(f"  query-only (no prompt)  : {p2[i]}")
    print(f"  VQA 'describe' question : {p3[i]}")
    print()

captioning:   0%|          | 0/1 [00:00<?, ?it/s]/home/matei/miniconda3/envs/vlm/lib/python3.10/site-packages/transformers/generation/utils.py:1127: UserWarning: Passing `no_repeat_ngram_size` with `inputs_embeds` and without `input_ids` to `generate` will apply n-gram constraints only to newly generated tokens, not to the prompt.
  warnings.warn(
captioning: 100%|██████████| 1/1 [00:08<00:00,  8.04s/it]


[ROCOv2_2023_test_000001]
  REF                     : CT chest axial view showing a huge ascending aortic aneurysm (*).
  query-only 'a photo of' : the inside of a patient's abdomen
with a white object in the middle
of the image

This content is reviewed regularly and is updated when new and relevant evidence is made available. This information
  query-only (no prompt)  : a view of the left side of the abdomen showing the liver, gallbladder, pancreas, and spleen
  VQA 'describe' question : Right lobe of the pancreas
This is an image of a patient's abdomen

This content is reviewed regularly and is updated when new and relevant evidence is made available. This information is neither

[ROCOv2_2023_test_000002]
  REF                     : Computed tomography (CT) shows floating thrombosis (white arrow)
  query-only 'a photo of' : the left side of the abdomen
with an arrow pointing to the right
side of the image
  query-only (no prompt)  : a black and white image of a man's chest
with an a

In [ ]:
# ── Caption metrics: BLEU-1..4, METEOR, ROUGE-L, CIDEr, BERTScore (all Java-free) ──
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bertscore

def _norm(s):
    return " ".join(str(s).lower().split())

def compute_caption_metrics(preds, refs):
    assert len(preds) == len(refs)
    # A model can emit a caption that is empty after decoding; BERTScore's
    # empty-string branch then crashes. Replace empties with "." (same guard as
    # caption_roco.py) so every scorer stays happy.
    preds = [p if str(p).strip() else "." for p in preds]
    refs  = [r if str(r).strip() else "." for r in refs]
    gts = {i: [_norm(refs[i])]  for i in range(len(refs))}
    res = {i: [_norm(preds[i])] for i in range(len(preds))}
    bleu, _  = Bleu(4).compute_score(gts, res)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    meteor = sum(meteor_score([_norm(refs[i]).split()], _norm(preds[i]).split())
                 for i in range(len(preds))) / len(preds)
    _, _, F = bertscore(preds, refs, lang="en", verbose=False)
    m = {"BLEU-1": bleu[0], "BLEU-2": bleu[1], "BLEU-3": bleu[2], "BLEU-4": bleu[3],
         "METEOR": meteor, "ROUGE-L": rouge, "CIDEr": cider, "BERTScore-F1": F.mean().item()}
    print("=" * 40)
    for k, v in m.items():
        print(f"  {k:14s}: {v:.4f}")
    print("=" * 40)
    return m

In [13]:
# ── Zero-shot captioning evaluation on ROCOv2 test ──
# Start with a subset for a quick number; set SUBSET_N = None for the full test set
# (~9.9k images -> tens of minutes; run it in tmux if you want the full figure).
SUBSET_N = 500

eval_recs = records if SUBSET_N is None else records[:SUBSET_N]
print(f"captioning {len(eval_recs)} images (prompt = {CAPTION_PROMPT!r}) ...")
preds = caption_records(eval_recs, prompt=CAPTION_PROMPT)
refs  = [r["caption"] for r in eval_recs]

print("\nZERO-SHOT CAPTIONING -- VQA-RAD (LoRA-ViT) model on ROCOv2 test")
metrics = compute_caption_metrics(preds, refs)

print("\nexamples:")
for r, p in list(zip(eval_recs, preds))[:5]:
    print(f"  REF : {r['caption'][:100]}")
    print(f"  PRED: {p[:100]}\n")

captioning 500 images (prompt = 'a photo of') ...


captioning:   0%|          | 0/63 [00:00<?, ?it/s]/home/matei/miniconda3/envs/vlm/lib/python3.10/site-packages/transformers/generation/utils.py:1127: UserWarning: Passing `no_repeat_ngram_size` with `inputs_embeds` and without `input_ids` to `generate` will apply n-gram constraints only to newly generated tokens, not to the prompt.
  warnings.warn(
captioning: 100%|██████████| 63/63 [08:33<00:00,  8.15s/it]



ZERO-SHOT CAPTIONING -- VQA-RAD (LoRA-ViT) model on ROCOv2 test
{'testlen': 9731, 'reflen': 10635, 'guess': [9731, 9231, 8731, 8231], 'correct': [1535, 282, 34, 7]}
ratio: 0.914997649271188


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2751.52it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  BLEU-1        : 0.1437
  BLEU-2        : 0.0633
  BLEU-3        : 0.0242
  BLEU-4        : 0.0102
  METEOR        : 0.0933
  ROUGE-L       : 0.1183
  CIDEr         : 0.0375
  BERTScore-F1  : 0.8278

examples:
  REF : CT chest axial view showing a huge ascending aortic aneurysm (*).
  PRED: the inside of a patient's abdomen
with a white object in the middle
of the image

This content is re

  REF : Computed tomography (CT) shows floating thrombosis (white arrow)
  PRED: the left side of the abdomen
with an arrow pointing to the right
side of the image

  REF : Digitally subtracted angiogram demonstrates active extravasation of the superior rectal artery into 
  PRED: a blue arrow on a white background
image 1 of 2 images showing the left side of the thoracic aorta i

  REF : Digitally subtracted angiogram of the IMA demonstrated cessation of flow through the proximal superi
  PRED: a black and white image of a car
with a car in the background

http://www.siriusxm.com/sites/default

  